In [5]:
from google.colab import drive
import pandas as pd
import sqlite3
import os


In [4]:
#Mount drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# defining path
project_path='/content/drive/MyDrive/customer-intelligence-pipeline'
raw_data_path = os.path.join(project_path, 'data/raw')
db_path = os.path.join(project_path, 'customer_analytics.db') # Path for our new database


In [7]:
# Load the data from CSV
customers_df = pd.read_csv(os.path.join(raw_data_path, 'customers.csv'))
orders_df = pd.read_csv(os.path.join(raw_data_path, 'orders.csv'))


In [8]:
#converting date columns to the correct format

orders_df['order_date']=pd.to_datetime(orders_df['order_date'])
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'])


In [9]:
orders_df.info()
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     5000 non-null   object        
 1   customer_id  5000 non-null   object        
 2   order_date   5000 non-null   datetime64[ns]
 3   amount       5000 non-null   float64       
 4   status       5000 non-null   object        
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 195.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       1000 non-null   object        
 1   signup_date       1000 non-null   datetime64[ns]
 2   segment           1000 non-null   object        
 3   region            1000 non-null   object        
 4   email_subscribed  1000 non-null   bool      

In [10]:
# 4. Create SQLite Database & Load Tables
print(" Creating SQLite Database...")
conn = sqlite3.connect(db_path)


 Creating SQLite Database...


In [11]:
# Write DataFrames to SQL tables
customers_df.to_sql('customers', conn, if_exists='replace', index=False)
orders_df.to_sql('orders', conn, if_exists='replace', index=False)

print(f" Database created at: {db_path}")
print(" Tables 'customers' and 'orders' are ready for SQL queries!")



 Database created at: /content/drive/MyDrive/customer-intelligence-pipeline/customer_analytics.db
 Tables 'customers' and 'orders' are ready for SQL queries!


In [12]:
conn.close()


In [13]:
# Basic SQL Exploration
conn = sqlite3.connect(db_path)

def run_query(query):
    return pd.read_sql(query, conn)


In [14]:
# 1. Count total customers and orders
print("--- Total Counts ---")
print(run_query("SELECT COUNT(*) as total_customers FROM customers"))
print(run_query("SELECT COUNT(*) as total_orders FROM orders"))


--- Total Counts ---
   total_customers
0             1000
   total_orders
0          5000


In [15]:
# 2. Check distinct segments
print("\n--- Customer Segments ---")
query_segments = """
SELECT segment, COUNT(*) as count
FROM customers
GROUP BY segment
ORDER BY count DESC;
"""
print(run_query(query_segments))



--- Customer Segments ---
      segment  count
0    Standard    613
1     Premium    287
2  Enterprise    100


In [16]:
# 3. Check total sales by status
print("\n--- Sales by Order Status ---")
query_sales = """
SELECT status, COUNT(*) as count, SUM(amount) as total_revenue
FROM orders
GROUP BY status;
"""
print(run_query(query_sales))



--- Sales by Order Status ---
      status  count  total_revenue
0  Cancelled    233       79104.72
1  Completed   4519     1544753.78
2   Returned    248       84278.49


In [17]:
# RFM Calculation using SQL

print("--- RFM Analysis Calculation ---")

rfm_query = """
WITH customer_rfm AS (
    SELECT
        customer_id,
        -- Recency: Days since last order (assuming today is the max order date in dataset)
        CAST(julianday((SELECT MAX(order_date) FROM orders)) - julianday(MAX(order_date)) AS INTEGER) as recency_days,

        -- Frequency: Count of orders
        COUNT(order_id) as frequency,

        -- Monetary: Total spend (only completed orders)
        SUM(amount) as monetary_value

    FROM orders
    WHERE status = 'Completed' -- Filter for real sales only
    GROUP BY customer_id
)

SELECT *
FROM customer_rfm
ORDER BY monetary_value DESC
LIMIT 10;
"""

rfm_df = run_query(rfm_query)
print(rfm_df)



--- RFM Analysis Calculation ---
  customer_id  recency_days  frequency  monetary_value
0  CUST_00321            74          7        11193.57
1  CUST_00451             2          5         7859.90
2  CUST_00769             9          5         7046.88
3  CUST_00493             8          6         7029.02
4  CUST_00973            60          6         6618.44
5  CUST_00277            11          8         6367.92
6  CUST_00852            15          8         6236.46
7  CUST_00381            13          6         6072.37
8  CUST_00052            85          5         6015.02
9  CUST_00522           337         13         5984.67


In [18]:
# Save RFM Analysis to CSV

full_rfm_query = """
SELECT
    customer_id,
    CAST(julianday((SELECT MAX(order_date) FROM orders)) - julianday(MAX(order_date)) AS INTEGER) as recency_days,
    COUNT(order_id) as frequency,
    SUM(amount) as monetary_value
FROM orders
WHERE status = 'Completed'
GROUP BY customer_id;
"""

full_rfm_df = run_query(full_rfm_query)

# Save to processed data folder
processed_path = os.path.join(project_path, 'data/processed', 'rfm_analysis.csv')
full_rfm_df.to_csv(processed_path, index=False)

print(f" RFM Analysis saved to: {processed_path}")
print(f"   Total records: {len(full_rfm_df)}")

conn.close()


 RFM Analysis saved to: /content/drive/MyDrive/customer-intelligence-pipeline/data/processed/rfm_analysis.csv
   Total records: 983
